In [ ]:
import requests
import numpy as np
import time
import tqdm
import tqdm.notebook
from kiss_headers import parse_it, get_polymorphic, SetCookie

rng = np.random.default_rng()

all_user_cookies = {}

def create_user(user):
      q = requests.post("http://localhost:9663/signup", 
                        allow_redirects=False,
                        headers={"Origin": "http://localhost:9663"},
                        data={
                              "username": user,
                              "password": user,
                              "email": f"{user}@wp.pl",
                              "agreement.assistance": "true",
                              "agreement.nice": "true",
                              "agreement.account": "true",
                              "agreement.policy": "true",
                              })
      headers = parse_it(q)
      if "Set-Cookie" in headers:
            for set_cookie in headers["Set-Cookie"]:
                  sc = get_polymorphic(set_cookie, SetCookie)
                  if sc.get_cookie_name() == "lila2":
                        return {"lila2": sc.get_cookie_value()}
      return None

def get_first_puzzle_id(user_cookies):
    r = requests.get("http://localhost:9663/training", cookies=user_cookies)
    pattern = '"id":"p'
    i = r.text.find(pattern)
    first = r.text[i+len(pattern)-1:i+len(pattern)+4]
    return first

def submit_puzzle(puzzle_id, solved, user_cookies):
    r = requests.post(f"http://localhost:9663/training/complete/mix/{puzzle_id}", 
                headers={"Origin": "http://localhost:9663"},
                data={"win": "true" if solved else "false", "rated": "true"},
                cookies=user_cookies,
                )
    return r.json()["next"]["puzzle"]["id"]

def solve_for_user(username, count):
    if username not in all_user_cookies:
        create_result = create_user(username)
        if create_result is None:
            raise Exception(f"cannot create {username}")
        all_user_cookies[username] = create_result
    user_cookies = all_user_cookies[username]
    puzzle_id = get_first_puzzle_id(user_cookies)
    for i in range(count):
        # print(f"{i+1}/{count} - {puzzle_id}")
        puzzle_id = submit_puzzle(puzzle_id, rng.integers(2), user_cookies)
        # time.sleep(0.01)

In [ ]:
for i in tqdm.tqdm(range(0, 20)):
    solve_for_user(f"user{i}", 500)

In [ ]:
import pymongo
import pandas as pd


client = pymongo.MongoClient()

all_puzzle_ids = [k["_id"] for k in client.lichess.puzzle2_puzzle.find({}, {"_id": 1})]

puzzle_round_stats = []
for puzzle_round in client.lichess.puzzle2_round.find():
    user, puzzle_id = puzzle_round["_id"].split(":")
    puzzle_round_stats.append({"puzzle_id": puzzle_id, "user": user, "w": puzzle_round["w"]})

df = pd.DataFrame(puzzle_round_stats)




In [ ]:
df.groupby("puzzle_id")["user"].count().hist()

In [ ]:
df.groupby("puzzle_id")["user"].count().sum()

In [ ]:
import matplotlib.pyplot as plt

vals = df.groupby("puzzle_id")["user"].count().to_numpy()
plt.scatter(range(len(vals)), vals)
